# Analyse predictive value of Sepsis-3 Components

In [ ]:
# Imports:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import shap
from lifelines import KaplanMeierFitter

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

import xgboost as xgb
from xgboost import XGBClassifier

from tableone import TableOne

from experiment_config import *
from utils import *
%matplotlib inline

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

## 1. Read table

In [ ]:
dataset = "mimiciv"
data_path = f"./filtered_data/{dataset}_tabledata_imputed_sepsis3.csv"

df = pd.read_csv(data_path)
df

In [ ]:
# Experiment parameters:
TEST_SPLIT_FRAC = 0.15

# Label translation dictionary:
label_dict = {
    # Demographics
    'gender': 'Male',
    'admission_age': 'Admission age',
    'weight_admit': 'Weight',
    'race_cat': 'Ethnicity',
    'race_cat_ASIAN': 'Asian (ethnicity)',
    'race_cat_BLACK': 'Black (ethnicity)',
    'race_cat_HISPANIC': 'Hispanic (ethnicity)',
    'race_cat_OTHER': 'Other (ethnicity)',
    'race_cat_UNKNOWN': 'Unknown (ethnicity)',
    'race_cat_WHITE': 'White (ethnicity)',
    'gcs_binned_Severe (3-8)': 'Severe GCS score (3-8)',
    'gcs_binned_Moderate (9-12)': 'Moderate GCS score (9-12)',
    'gcs_binned_Mild (13-15)': 'Mild GCS score (13-15)',

    # Vasopressor Usage
    'vp_last6h': 'Documented vasopressor use (last 6h)',

    # Comorbidities
    'sepsis3': 'Sepsis',
    'myocardial_infarct': 'Myocardial infarct',
    'congestive_heart_failure': 'Congestive heart failure',
    'peripheral_vascular_disease': 'Peripheral vascular disease',
    'cerebrovascular_disease': 'Cerebrovascular disease',
    'chronic_pulmonary_disease': 'COPD',
    'liver_disease': 'Liver disease',
    'renal_disease': 'Renal disease',
    'malignant_cancer': 'Malignant cancer',
    'diabetes': 'Diabetes',
    
    # Scores
    'gcs_binned': 'Glascow Coma Score',
    'apsiii': 'APS-III score',

    # Vital signs
    'heart_rate_mean_last24h': 'Mean heart rate',
    'mbp_mean_last24h': 'Mean MBP',
    'glucose_mean_last24h': 'Mean glucose',
    'platelet_last': 'Mean platelet',
    'spo2_mean_last24h': 'Mean SpO2',
    'resp_rate_mean_last24h': 'Mean respiratory rate',
    'temperature_mean_last24h': 'Mean temperature',
    'urineoutput_24hr': 'Urine output (ml)',
    'fluidbalance_24hr': 'Fluid balance (ml)',
    
    # Lab values and blood gasses
    'bicarbonate_last': 'Bicarbonate',
    'po2_last': 'pO2',
    'ptt_last': 'PTT',
    'inr_last': 'INR',
    'calcium_last': 'Calcium',
    'potassium_last': 'Potassium',
    'mchc_last': 'MCHC',
    'mch_last': 'MCH',
    'ph_last': 'pH',
    'aniongap_last': 'Anion Gap',
    'sodium_last': 'Sodium',
    'hemoglobin_last': 'Hemoglobin',
    'wbc_last': 'WBC',
    'rdw_last': 'RDW',
    'creatinine_last': 'Creatinine',

    # Settings
    'max_flow_rate': 'Highest flow-rate',

    # Secondary outcomes
    'inhosp_mortality': 'In-hospital mortality',

    # Sepsis subscores
    "sofa_renal": 'Renal (SOFA score)',
    "sofa_cns": 'Central Nervous System (SOFA score)',
    "sofa_cardiovascular": 'Cardiovascular (SOFA score)',
    "sofa_liver": 'Liver (SOFA score)',
    "sofa_coagulation": 'Coagulation (SOFA score)',
    "sofa_respiration": 'Respiration (SOFA score)',
}

In [ ]:
print(f"Number (%) of sepsis patients: {len(df[df.sepsis3==1])}/{len(df)} ({(len(df[df.sepsis3==1])/len(df)) * 100}%)")

### Set which group to run for:
1. Sepsis     --> Just the sepsis patients
2. NonSepsis  --> Just the patients without sepsis
3. All        --> All patients, using just the SOFA scores and most important features

In [ ]:
analysis_type = "NonSepsis" # ["Sepsis", "NonSepsis", "All"]

In [ ]:
data = df.iloc[:, 1:]

# Experiment with conditioning on sepsis subpopulations:

if analysis_type == "Sepsis":
    data = data[data.sepsis3 == 1]
elif analysis_type == "NonSepsis":
    data = data[data.sepsis3 == 0]
else:
    data = data

data = data[[
    # "intubated",
    "hfno_failure",

    # "sepsis3",

    # "s3_sofa_score",
    # "s3_sofa_time",
    # "s3_antibiotic_time",
    
    # "s3_renal",
    # "s3_cns",
    # "s3_cardiovascular",
    # "s3_liver",
    # "s3_coagulation",
    # "s3_respiration",

    "sofa_renal",
    "sofa_cns",
    "sofa_cardiovascular",
    "sofa_liver",
    "sofa_coagulation",
    "sofa_respiration",
     
    "apsiii",
    # 'admission_age',
    'spo2_mean_last24h',
    'weight_admit',
    # 'diabetes',
    # 'urineoutput_24hr'
    # 'fluidbalance_24hr'
    
]]



data

## 1.1 Split data into training and test dataset

In [ ]:
X = data.iloc[:,1:].to_numpy()
y = data.iloc[:,0].to_numpy()
feature_names = data.columns[1:].tolist()

for i, col in enumerate(feature_names):
    if col in label_dict.keys():
        feature_names[i] = label_dict[col]

print(X)
print(y)

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)
# print(f"Proportion intubated in training dataset: {np.mean(y_train)}")
# print(f"Proportion intubated in test dataset: {np.mean(y_test)}")

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
# print(f"Proportion intubated in training dataset: {np.mean(y_train)}")
# print(f"Proportion intubated in test dataset: {np.mean(y_test)}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"Proportion intubated in training dataset: {np.mean(y_train)}")
print(f"Proportion intubated in test dataset: {np.mean(y_test)}")

In [ ]:
X_train

## 2. Perform Logistic Regression

In [ ]:
# Scale the data before fitting model:
scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_train_scaled

scaler_fullset = preprocessing.StandardScaler().fit(X)
X_scaled = scaler_fullset.transform(X)
X_scaled

In [ ]:
# Using cross validation:
clf = LogisticRegression()

scores = cross_validate(clf, X_scaled, y,
    cv=10,
    scoring=(
        'accuracy',
        'precision',
        'recall',
        'f1',
        'roc_auc'
    ),
    return_train_score=True,
    return_estimator=True
)

scores

In [ ]:
# Using training and test splits:
clf2 = LogisticRegression(
    class_weight='balanced',
    max_iter=1000000
)

clf2.fit(X_train, y_train)
preds = clf2.predict(X_test)
probs = clf2.predict_proba(X_test)
print(preds)
print(y_test)
print(metrics.classification_report(preds, y_test))

In [ ]:
cm = metrics.confusion_matrix(y_test, preds)
score = clf2.score(X_test, y_test)
# auroc_score = metrics.roc_auc_score(y_test, preds)
auroc_score = metrics.roc_auc_score(y_test, probs[:, 1])

# Source: https://towardsdatascience.com/logistic-regression-using-python-sklearn-numpy-mnist-handwriting-recognition-matplotlib-a6b31e2b166a
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
plt.ylabel('Actual label');
plt.xlabel('Predicted label');
all_sample_title = 'Acc: {0}, AUROC: {1}'.format(round(score, 4), round(auroc_score, 4))
plt.title(all_sample_title, size = 15);

In [ ]:
shap.initjs()


explainer = shap.Explainer(clf2, X_train, feature_names=feature_names)
shap_values = explainer(X_test)
# shap.plots.beeswarm(shap_values)

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
print(f"Intercept: {clf2.intercept_}")
pd.DataFrame(zip(data.columns[1:].tolist(), clf2.coef_[0]))
# clf.coef_

## 3. Apply XGBoost

In [ ]:
balance_ratio = (1 - np.mean(y_train)) / np.mean(y_train)
print(balance_ratio)

# Source: https://machinelearningmastery.com/configure-gradient-boosting-algorithm/
# xgb_model = xgb.XGBClassifier(
#     objective='binary:logistic',
#     random_state=42,
#     scale_pos_weight=balance_ratio,
#     max_depth=6,
#     # eta=0.01,
#     reg_alpha=0,
#     reg_lambda=1,
#     min_child_weight=3,
#     n_estimators=150,
#     learning_rate=0.01,
#     subsample=0.5,
#     # max_delta_step=0.01
# )
# xgb_model = xgb.XGBClassifier(
#     objective='binary:logistic',
#     random_state=42,
#     scale_pos_weight=balance_ratio,
#     # scale_pos_weight=1,
#     max_depth=15,
#     # eta=0.01,
#     reg_alpha=0,
#     reg_lambda=1,
#     min_child_weight=6,
#     n_estimators=150,
#     learning_rate=0.01,
#     subsample=0.5,
#     # max_delta_step=0.01
# )

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    scale_pos_weight=balance_ratio,
    # scale_pos_weight=1,
    max_depth=7,
    # eta=0.01,
    reg_alpha=0,
    reg_lambda=1,
    min_child_weight=6,
    n_estimators=350,
    learning_rate=0.05,
    subsample=0.5,
    # max_delta_step=0.01
)

eval_set = [(X_train, y_train), (X_test, y_test)]
# eval_set = [(X_train, y_train)]

xgb_model.fit(
    X_train,
    y_train,
    eval_set=eval_set,
    verbose=True
)

xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)
print(xgb_preds)
print(y_test)

# xgb_model, xgb_preds = mc_bootstrap(X_train, y_train, X_test, y_test, xgb_model, model_label='XGBoost', eval_set=eval_set, verbose=True, n_iter=N_ITER_BOOTSTRAP)


In [ ]:
xgb_cm = metrics.confusion_matrix(y_test, xgb_preds)
xgb_score = xgb_model.score(X_test, y_test)
# xgb_auroc_score = metrics.roc_auc_score(y_test, xgb_preds)
xgb_auroc_score = metrics.roc_auc_score(y_test, xgb_probs[:, 1])
# xgb_auroc_score = metrics.roc_auc_score(xgb_preds, y_test)

# Source: https://towardsdatascience.com/logistic-regression-using-python-sklearn-numpy-mnist-handwriting-recognition-matplotlib-a6b31e2b166a
plt.figure(figsize=(6,6))
sns.heatmap(xgb_cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
plt.ylabel('Actual label');
plt.xlabel('Predicted label');
all_sample_title = 'Acc: {0}, AUROC: {1}'.format(round(xgb_score, 4), round(xgb_auroc_score, 4))
plt.title(all_sample_title, size = 15);

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
xgb_explainer = shap.Explainer(xgb_model, X_train, feature_names=feature_names)
xgb_shap_values = xgb_explainer(X_test, check_additivity=False)

In [ ]:
shap.summary_plot(xgb_shap_values, X_test, show=False)

plt.savefig(f'results/{analysis_type}_sepsis3_separated_analysis_SHAP_{round(xgb_auroc_score,2)}AUROC.pdf', bbox_inches='tight')

In [ ]:
# xgb_preds_man = (xgb_probs[:, 1] >= 0.7).astype(int)
xgb_cm = metrics.confusion_matrix(y_test, xgb_preds)
# xgb_cm = metrics.confusion_matrix(y_test, xgb_preds_man)
xgb_score = xgb_model.score(X_test, y_test)
# xgb_auroc_score = metrics.roc_auc_score(y_test, xgb_probs[:, 1])
xgb_prauc_score = metrics.average_precision_score(y_test, xgb_probs[:, 1])


# Source: https://towardsdatascience.com/logistic-regression-using-python-sklearn-numpy-mnist-handwriting-recognition-matplotlib-a6b31e2b166a
plt.figure(figsize=(6,6))
sns.heatmap(xgb_cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
plt.ylabel('Actual label');
plt.xlabel('Predicted label');
all_sample_title = 'Acc: {0}, AUPRC: {1}'.format(round(xgb_score, 4), round(xgb_prauc_score, 4))
plt.title(all_sample_title, size = 15);

In [ ]:
# for i, val in enumerate(xgboost_model.feature_importances_):
#     if val < 0.03:
#         print(f"{data.columns[1+i]}: {val}")
#     else:
#         print(f"*** --> {data.columns[1+i]}: {val}")

In [ ]:
# from xgboost import plot_importance
# plot_importance(xgboost_model, max_num_features=10)

In [ ]:
# feature_ids = [13, 11, 40, 18, 16, 42, 15, 19, 17, 20]
# for id in feature_ids:
#     print(data.columns[1+id])

In [ ]:
# for k,v in xgboost_model.get_booster().get_score(importance_type='weight').items():
#     print(f"{data.columns[1+int(k[1:])]}: {v}")